# Conformal filtering

First calibrate a recall threshold for the candidate set. Then calibrate a precision threshold within those candidates. Candidates split into confident and uncertain detections; the remaining detections are declined.

## Load the data

Place the downloaded files in `data/`, or set `GFLD_DATA_DIR` to their location. Dataset details are provided in the paper.

In [ ]:
%matplotlib inline
from pathlib import Path
from dataclasses import asdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from gfld import *

# Works when Jupyter starts in the repository root or notebooks directory.
ROOT = Path.cwd() if (Path.cwd() / "pyproject.toml").is_file() else Path.cwd().parent
DATA_DIR = example_directory(ROOT)  # GFLD_DATA_DIR or repository data/
data = load_example(DATA_DIR)
print({k: v["images"] for k, v in data["metadata"]["splits"].items()})

In [ ]:
# Split calibration and evaluation by image.
_, calibration_idx, evaluation_idx = next(calibration_splits(
    len(data["internal"]), folds=5, permutations=1, seed=42))
calibration = [data["internal"][i] for i in calibration_idx]
evaluation = [data["internal"][i] for i in evaluation_idx]
annotations = data["internal_annotations"]
assert set(p.image_id for p in calibration).isdisjoint(p.image_id for p in evaluation)
print(f"{len(calibration)} calibration images; {len(evaluation)} held-out images")

In [ ]:
cp = calibrate_cp(calibration, annotations, alpha=0.01)
pd.DataFrame([asdict(cp)])

## Calibrate the two thresholds

Set the recall and precision targets, then calibrate their confidence thresholds.

In [ ]:
filtering = calibrate_filtering(calibration, annotations, cp,
    target_recall=0.80, target_precision=0.85, delta=0.5)
pd.DataFrame([asdict(filtering.recall), asdict(filtering.precision)],
             index=["recall", "precision"])

## Inspect the threshold curves

Compare the calibration bounds with the requested targets. Precision is calibrated within the recall candidate set.

In [ ]:
from gfld.filtering import detection_counts, binomial_lower
counts = detection_counts(calibration, annotations, cp)
candidates = [p.select(p.scores >= filtering.recall.threshold) for p in calibration]
precision_counts = detection_counts(candidates, annotations, cp)
thresholds = np.linspace(0, 1, 81)
recall_bounds, precision_bounds = [], []
for t in thresholds:
    keep = counts.scores >= t
    k = int(counts.is_tp[keep].sum())
    recall_bounds.append(binomial_lower(k, int(counts.gt_counts.sum()), 0.5))
    keep_p = precision_counts.scores >= t
    k_p = int(precision_counts.is_tp[keep_p].sum())
    precision_bounds.append(binomial_lower(k_p, int(keep_p.sum()), 0.5)
                            if t >= filtering.recall.threshold else np.nan)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for ax, values, target, threshold, label in zip(axes,
        [recall_bounds, precision_bounds], [0.80, 0.85],
        [filtering.recall.threshold, filtering.precision.threshold], ["Recall", "Precision"]):
    ax.plot(thresholds, values); ax.axhline(target, color="black", linestyle="--")
    ax.axvline(threshold, color="tab:red")
    ax.set(xlabel="Confidence threshold", ylabel=f"{label} lower bound", ylim=(0, 1))
plt.tight_layout(); plt.show()

## Apply and evaluate

`apply` needs only predictions. Evaluation requires annotations. Recall belongs to the combined candidate set; precision belongs to the confident set.

In [ ]:
regions = filtering.apply(evaluation)
metrics = evaluate_filtering(evaluation, annotations, filtering)
display(pd.DataFrame({k: metrics[k] for k in ("baseline", "candidate", "confident")}).T)
plt.bar(metrics["regions"].keys(), metrics["regions"].values())
plt.ylabel("Detections"); plt.show()

## Repeated cross-calibration

Run five folds over four permutations. Pool counts across evaluation folds before computing each permutation’s metrics.

In [ ]:
RUN_FULL_CROSS_VALIDATION = False
if RUN_FULL_CROSS_VALIDATION:
    repeated = cross_validate(data["internal"], annotations, folds=5, permutations=4, seed=42)
    display(pd.DataFrame(repeated["permutations"]))
    print(repeated["mean"])

Recall calibration retains all candidates when its target is infeasible. Precision calibration assigns no confident detections when no threshold passes. The returned status records these cases.